# 03. 계약서 → 엔진 → NPV

계약서(무엇)와 평가 방법(어떻게)을 나눈다(원칙 6). 계약서는 순수 데이터이고, 엔진은 팩토리에서 받는다.

| 단계 | QuantTool 코드 | ORE·QuantLib 원본 |
|---|---|---|
| 계약서 | `positions/ust_book.yaml`, `src/quanttool/derivatives/rates.py` `FixedBond` | `OREData/ored/portfolio/trade.hpp`, `bond.hpp` |
| 엔진 선택 | `configs/pricing/engines.yaml` | `Examples/*/Input/pricingengine.xml` |
| 팩토리·빌더 | `src/quanttool/derivatives/factory.py`, `engines.py` | `OREData/ored/portfolio/enginefactory.hpp`, `builders/bond.hpp` |
| 상품↔엔진 대화 | — | `docs/references/quantlib.md` 3절 |

> 데이터: `tests/fixtures/ust_par_sample.csv`는 재무부 CSV와 **형식만 같은 합성 샘플**이다. 흐름을 익히는 용도이고, 숫자로 시장을 해석하지 않는다.

In [ ]:
from pathlib import Path

# 노트북이 어디서 열리든 레포 루트(pyproject.toml이 있는 곳)를 찾는다
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SAMPLE_CSV = ROOT / "tests" / "fixtures" / "ust_par_sample.csv"  # 합성 샘플 (실제 시세 아님)

from datetime import date

import pandas as pd
import QuantLib as ql

from quanttool.config import load_config
from quanttool.derivatives import EngineFactory, Portfolio
from quanttool.market import Snapshot, UstParCsvLoader

cfg = load_config(ROOT / "configs")
loader = UstParCsvLoader(SAMPLE_CSV)
snap = Snapshot.build(date(2026, 10, 23), cfg, loader)

## 1. 계약서는 데이터다
엔진도, 커브 객체도 모른다. 할인 커브는 **이름**(`discount: UST`)으로만 가리킨다.

In [ ]:
book = Portfolio.load(ROOT / "positions" / "ust_book.yaml")
pd.DataFrame([t.model_dump() for t in book]).set_index("id")

## 2. 조립: `trade.build(factory)`
`FixedBond.build`가 일정·관례로 `ql.FixedRateBond`를 만들고, `factory.engine_for(trade)`로 엔진을 받는다. 팩토리는 `engines.yaml`에서 `FixedBond → DiscountingBond`를 찾아 그 엔진 빌더에 맡긴다.

In [ ]:
factory = EngineFactory(snap, cfg)
book.build(factory)
bond10 = book.trades["UST_10Y"]
ql_bond = bond10.instrument.instrument  # 안쪽의 QuantLib 상품
pd.DataFrame(
    [(cf.date().ISO(), cf.amount()) for cf in ql_bond.cashflows()], columns=["date", "amount"]
).tail()

In [ ]:
with snap.active():
    display(book.npvs().round(2))
    print("합계", round(book.npv(), 2))

## 3. QuantLib으로 직접 계산한 값과 같은가

In [ ]:
with snap.active():
    sch = ql.Schedule(
        ql.Date(15, 8, 2026),
        ql.Date(15, 8, 2036),
        ql.Period(ql.Semiannual),
        ql.UnitedStates(ql.UnitedStates.GovernmentBond),
        ql.Unadjusted,
        ql.Unadjusted,
        ql.DateGeneration.Backward,
        False,
    )
    direct = ql.FixedRateBond(
        1,
        100.0,
        sch,
        [0.04125],
        ql.ActualActual(ql.ActualActual.Bond),
        ql.Unadjusted,
        100.0,
        ql.Date(15, 8, 2026),
    )
    direct.setPricingEngine(ql.DiscountingBondEngine(snap.yield_curve("UST")))
    print(direct.NPV() * 1e7 / 100, bond10.npv())

## 4. 엔진은 재료가 같으면 공유된다
같은 할인 커브를 쓰는 채권 4개는 엔진 하나를 같이 쓴다 (ORE `CachingPricingEngineBuilder` 대응).

In [ ]:
len({id(factory.engine_for(t)) for t in book})

## 5. 시세를 바꾸면 재조립 없이 NPV가 바뀐다 (00번 노트북의 원리)

In [ ]:
with snap.active():
    q = snap.quotes["UST/PAR/10Y"]
    base = bond10.npv()
    q.setValue(q.value() + 0.0001)
    print("10Y 시세 +1bp → UST_10Y 손익", round(bond10.npv() - base, 2))
    q.setValue(q.value() - 0.0001)

## 6. 같은 계약서, 다른 날짜의 시장
계약서는 그대로 두고 다른 시장으로 다시 조립한다.

In [ ]:
snap_prev = Snapshot.build(date(2026, 10, 22), cfg, loader)
with snap_prev.active():
    book.build(EngineFactory(snap_prev, cfg))
    prev = book.npvs()
with snap.active():
    book.build(EngineFactory(snap, cfg))
    now = book.npvs()
pd.DataFrame({"10/22": prev, "10/23": now, "손익": now - prev}).round(2)

다음: `04_scenarios.ipynb` — 시장을 흔들어 민감도와 시나리오 손익을 본다.